# 4.1 and 4.5 Model comparison and shortlist

Owner: Fidan Shiraliyeva

The first part of this notebook repeats the section 3 pipeline from `03_prepare_the_data.ipynb`, so it runs on its own. The model comparison starts at **4.1 Model comparison** below.

## 0. Setup

Starting from the team's final table (85,371 orders), the training period is orders purchased before 1 April 2018 (64,038 orders, split decided by the team, DECISIONS D6).

Section 3 only uses the training data. The test set (orders from April 2018 onward) is not touched here and is evaluated at the end of the project.

The model inputs are the 22 approval-time features. IDs are excluded because they carry no meaning, raw dates are excluded because their information is already in the engineered timing features (plus the delivery date would be a data leakage), and is_late is of course excluded as this is the target variable

In [2]:
import pandas as pd

df = pd.read_csv("../data/processed/olist_final.csv",
                 parse_dates=["order_purchase_timestamp"])
df = df.sort_values("order_purchase_timestamp").reset_index(drop=True)

# Training period only: orders purchased before 1 April 2018 (the team's split)
train = df[df["order_purchase_timestamp"] < "2018-04-01"]

# Features = everything except IDs, raw dates and the target
non_features = ["order_id", "customer_id", "order_purchase_timestamp", "order_approved_at",
                "order_delivered_customer_date", "order_estimated_delivery_date", "is_late"]
X_train = train.drop(columns=non_features)
y_train = train["is_late"]
print(X_train.shape)

(64038, 22)


## 3.1 Cleaning

### 3.1a Missing Values

Five columns have missing values in the 64,038 training orders (primary product category, max distance, mean distance, max product weight and mean product weight).

Of these 5 columns, the product category column has the most missing values (about 1.83% rows total). The distance columns are both missing about 0.52% of rows, and the product weight columns are missing only about 0.02%.

In [3]:
missing = X_train.isna().sum()
missing = missing[missing > 0].sort_values(ascending=False)
pd.DataFrame({"rows_missing": missing,
              "percent": (missing / len(X_train) * 100).round(2)})

,rows_missing,percent
primary_product_category,1169,1.83
max_distance_km,330,0.52
mean_distance_km,330,0.52
mean_product_weight_g,16,0.02
max_product_weight_g,16,0.02


Dropping rows would remove real orders, so we are filling the missing rows instead of removing them. The empty rows in the numerical columns are replaced with the median values, and the empty rows in the category column are replaced with "unknown".

**The below step is shown on a copy of the table. The same filling is applied inside the pipeline (3.4), so it is fit on the training data only.**

Additionally, the distance columns will also get a "was missing" flag in the pipeline (3.4). This is because a missing distance means the ZIP code was not found in the original geolocation table, which might point to strange addresses (and could contribute to why an order was delivered late).

In [4]:
dist_cols = ["mean_distance_km", "max_distance_km"]
weight_cols = ["mean_product_weight_g", "max_product_weight_g"]

X_train_filled = X_train.copy()

for c in dist_cols:
    X_train_filled[c + "_was_missing"] = X_train_filled[c].isna().astype(int)

fill_cols = dist_cols + weight_cols
X_train_filled[fill_cols] = X_train_filled[fill_cols].fillna(X_train_filled[fill_cols].median())
X_train_filled["primary_product_category"] = X_train_filled["primary_product_category"].fillna("unknown")

print(X_train_filled.isna().sum().sum())

0


### 3.1b Outliers

The below table shows which columns have extreme values (potential outliers).

In [5]:
X_train.select_dtypes("number").skew().sort_values(ascending=False).round(1)

,0
total_price,11.1
seller_count,10.7
total_freight,8.7
item_count,8.2
distinct_product_count,7.7
total_product_volume_cm3,7.7
approval_lag_hours,5.2
freight_to_price_ratio,3.7
mean_product_weight_g,3.6
max_product_weight_g,3.6


#### No outliers are removed
We did not remove any outliers because the extreme values are all still real orders (i.e., 0 rows removed). No impossible values were found, so skewed columns are handled with a log transform and scaling instead

In [6]:
cols = ["total_price", "seller_count", "total_freight", "item_count",
        "distinct_product_count", "total_product_volume_cm3", "approval_lag_hours",
        "freight_to_price_ratio", "mean_product_weight_g", "max_product_weight_g"]

summary = X_train[cols].agg(["median", lambda s: s.quantile(0.99), "max"]).T
summary.columns = ["median", "99th percentile", "max"]
summary.round(1)


,median,99th percentile,max
total_price,85.0,978.0,13440.0
seller_count,1.0,2.0,5.0
total_freight,16.7,98.0,1002.3
item_count,1.0,3.0,21.0
distinct_product_count,1.0,2.0,7.0
total_product_volume_cm3,7744.0,140448.0,1476000.0
approval_lag_hours,0.3,87.4,741.4
freight_to_price_ratio,0.2,1.4,5.7
mean_product_weight_g,700.0,19100.0,40425.0
max_product_weight_g,700.0,19400.0,40425.0


### 3.1c Duplicates
There are no duplicate orders (every order_id in the 64,038 training orders appears only once) therefore 0 rows are removed

In [7]:
print(train["order_id"].is_unique)
print(train["order_id"].duplicated().sum())

True
0


## 3.2 Feature Engineering

### 3.2a Feature selection
Two features are dropped from the current features because they do not provide as much useful information: customer_zip_code_prefix and approval_month

customer_zip_code_prefix has 13,584 different values. One-hot encoding them would turn each value into its own column, which would make the table far too big and slow. As well, Customer state and distance columns already describe the locations.

approval_month is the same as purchase_month for 98.2% of orders. As well, the approval timing is already accounted for by the columns approval_lag_hours and promised_days_from_approval.

This leaves 20 features instead of 22.

In [8]:
print(X_train["customer_zip_code_prefix"].nunique())
print((X_train["approval_month"] == X_train["purchase_month"]).mean().round(3))

drop_cols = ["customer_zip_code_prefix", "approval_month"]
X_train_sel = X_train.drop(columns=drop_cols)
print(X_train_sel.shape)

13584
0.982
(64038, 20)


### 3.2b Feature groups

| Group | Features | Why it should help predict late deliveries |
|---|---|---|
| Timing | promised days, approval lag, purchase month, weekday and hour | Short promises, slow approvals and busy periods make delays more likely |
| Location | customer state, seller state | Some regions may be harder to deliver to |
| Distance | mean and max distance, same-state share | The farther a package travels, the longer it takes |
| Order size and cost | item count, seller count, total price, freight, freight-to-price ratio | Larger orders and orders from multiple sellers may be harder to ship on time |
| Product | weight, volume, category | Heavy or bulky products are slower to handle and ship |

### 3.2c Transformations
Discretizing continuous features is not necessary because the models can use the raw numbers. As well, date and time features are already split into month, weekday and hour.

The 13 columns with skew above 1 get a log(1 + x) transform, which makes the distribution more even by compressing the largest values (shrinks the long tails).

We note that log-transforming 3 of these columns did not help much (seller_count, distinct_product_count and item_count). However, keeping them logged is harmless and keeps the treatement consistent.

As well, for the two distance columns, the log transform over-corrects the skew, from +1.7 to −1.1. The skew is still lower than before, so the log transform is kept for these columns.

In [9]:
import numpy as np

log_cols = ["total_price", "seller_count", "total_freight", "item_count",
            "distinct_product_count", "total_product_volume_cm3", "approval_lag_hours",
            "freight_to_price_ratio", "mean_product_weight_g", "max_product_weight_g",
            "mean_distance_km", "max_distance_km", "promised_days_from_approval"]

before = X_train[log_cols].skew()
after = np.log1p(X_train[log_cols]).skew()
pd.DataFrame({"skew before": before, "skew after log": after}).round(1)

,skew before,skew after log
total_price,11.1,0.3
seller_count,10.7,9.8
total_freight,8.7,1.2
item_count,8.2,4.0
distinct_product_count,7.7,6.2
total_product_volume_cm3,7.7,-0.1
approval_lag_hours,5.2,1.0
freight_to_price_ratio,3.7,1.9
mean_product_weight_g,3.6,0.5
max_product_weight_g,3.6,0.4


## 3.3 Scaling and Encoding

### 3.3a Scaling
All numeric columns are standardized (mean 0, standard deviation 1) with StandardScaler, so that columns measured in very different units are on the same scale (e.g., price, weight in g, distance in km).

**The scaling is applied inside the pipeline (3.4), so it is fit on the training data only.**

In [10]:
from sklearn.preprocessing import StandardScaler

num_cols = X_train_sel.select_dtypes(include="number").columns
scaled = StandardScaler().fit_transform(X_train_sel[num_cols])
pd.DataFrame(scaled, columns=num_cols).agg(["mean", "std"]).round(2)

,promised_days_from_approval,approval_lag_hours,purchase_month,purchase_weekday,purchase_hour,item_count,distinct_product_count,seller_count,total_price,total_freight,mean_product_weight_g,max_product_weight_g,total_product_volume_cm3,mean_distance_km,max_distance_km,same_state_item_share,freight_to_price_ratio
mean,0.0,0.0,0.0,0.0,-0.0,0.0,-0.0,-0.0,0.0,-0.0,-0.0,-0.0,-0.0,0.0,0.0,-0.0,0.0
std,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0


### 3.3b Encoding
Three columns are text: customer_state (27 values), any_seller_state (22) and primary_product_category (71). The text columns are one-hot encoded, giving each value its own 0/1 column.

Categories with very few orders (fewer than 100 in the training data) are grouped into a single "infrequent" column instead of getting their own, because there is not enough data for the model to learn much from them.

Finally, the 1,169 missing product categories are filled with "unknown" before encoding, so they become their own category.

**The encoding is applied inside the pipeline (3.4), so it is fit on the training data only.**

In [11]:
cat_cols = X_train_sel.select_dtypes(exclude="number").columns
pd.DataFrame({"different_values": X_train_sel[cat_cols].nunique(),
              "rows_missing": X_train_sel[cat_cols].isna().sum()})

,different_values,rows_missing
customer_state,27,0
any_seller_state,22,0
primary_product_category,71,1169


## 3.4 Pipeline and Leakage Checks

### 3.4a Pipeline
All preprocessing is combined into one scikit-learn pipeline (ColumnTransformer), so every step is fit on the training data only. In cross-validation it is refit on each fold's training part, and the final fitted pipeline is then applied to the test set without refitting.

The steps are:
- Missing values: median for distance and weight, "unknown" for product category, plus a "was missing" flag for the distance columns.
- Log transform (log(1 + x)) on the 13 skewed numeric columns.
- Standardization of all numeric columns.
- One-hot encoding of the three text columns, with rare and unseen categories grouped as "infrequent".

After one-hot encoding the text columns, the pipeline produces 103 features.

In [12]:
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer, MissingIndicator
from sklearn.preprocessing import StandardScaler, OneHotEncoder, FunctionTransformer

dist_cols = ["mean_distance_km", "max_distance_km"]
weight_cols = ["mean_product_weight_g", "max_product_weight_g"]

# Skewed columns with no missing values: logged then scaled
log_cols = ["total_price", "seller_count", "total_freight", "item_count",
            "distinct_product_count", "total_product_volume_cm3",
            "approval_lag_hours", "freight_to_price_ratio", "promised_days_from_approval"]

# Skewed columns with missing values: filled, logged, then scaled
log_fill_cols = dist_cols + weight_cols

# Remaining numerical columns: only scaled
other_num = ["purchase_month", "purchase_weekday", "purchase_hour", "same_state_item_share"]

# Text columns
cat_cols = ["customer_state", "any_seller_state", "primary_product_category"]

# Logged then standardized
log_scale = Pipeline([("log", FunctionTransformer(np.log1p)),
                      ("scale", StandardScaler())])

# Medians filled, logged, then standardized
fill_log_scale = Pipeline([("fill", SimpleImputer(strategy="median")),
                           ("log", FunctionTransformer(np.log1p)),
                           ("scale", StandardScaler())])

# "unknown" filled, then one-hot (rare and unseen values become "infrequent")
cat_pipe = Pipeline([("fill", SimpleImputer(strategy="constant", fill_value="unknown")),
                     ("onehot", OneHotEncoder(handle_unknown="infrequent_if_exist",
                                              min_frequency=100))])

# Each step applied to its own columns
preprocess = ColumnTransformer([
    ("log_scale", log_scale, log_cols),
    ("fill_log_scale", fill_log_scale, log_fill_cols),
    ("dist_flag", MissingIndicator(features="all"), dist_cols),  # "was missing" flag
    ("scale", StandardScaler(), other_num),
    ("categories", cat_pipe, cat_cols),
], sparse_threshold=0)

# Check: fit on the training data only
X_pre = preprocess.fit_transform(X_train_sel)
print(X_pre.shape)
print(np.isnan(X_pre.astype(float)).sum())

(64038, 103)
0


### 3.4b Leakage Checks
Late delivery predictions are made at the moment an order is approved, so the model can only use information that is already available at that point.
The pipeline is also fit on the training data only so that no validation or test information reaches the fitted steps.

The table below shows which columns are known when the order is approved (no leakage) and which are only known afterwards, during or after delivery (leakage).

| Column | Known at approval? | Reason |
|---|---|---|
| Timing: promised_days_from_approval, approval_lag_hours, purchase_month, purchase_weekday, purchase_hour | Y | Come from the purchase time, approval time and the estimated delivery date, which the customer sees when ordering. |
| Location: customer_state, any_seller_state | Y | Customer and seller locations are known at checkout. |
| Distance: mean_distance_km, max_distance_km, same_state_item_share | Y | Calculated from the customer and seller ZIP codes, both known when the order is placed. |
| Order size and cost: item_count, distinct_product_count, seller_count, total_price, total_freight, freight_to_price_ratio | Y | The items ordered, their prices and the freight are fixed when the order is placed. |
| Product: mean_product_weight_g, max_product_weight_g, total_product_volume_cm3, primary_product_category | Y | Product catalogue attributes that exist before the order. |
| customer_zip_code_prefix, approval_month | Y (dropped) | Known at approval, but dropped for being too detailed (13,584 values) and nearly identical to purchase_month. |
| order_status, order_delivered_carrier_date, order_delivered_customer_date, review columns | N | Only known after approval, during or after delivery. |
| is_late | N | The target, which is only known after delivery. |


## 4.1 Model comparison

Below, we create a reusable evaluation framework for comparing the machine learning models that predict whether an order will be delivered late.

In [13]:
import time
import numpy as np
import pandas as pd

from sklearn.base import BaseEstimator, ClassifierMixin, clone
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder, FunctionTransformer
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier

RANDOM_STATE = 42


def recall_at_top(y_true, scores, frac):
    """Calculate the share of late orders captured in the top-risk group."""
    y_true = np.asarray(y_true)

    # Select the top frac proportion of orders with the highest risk scores.
    k = int(np.ceil(frac * len(y_true)))
    top = np.argsort(-scores, kind="stable")[:k]

    # Calculate what proportion of all actual late orders are in this group.
    return y_true[top].sum() / y_true.sum()


def get_scores(model, X):
    """Obtain continuous scores for ranking orders by late-delivery risk."""

    # Use predicted probabilities when available; otherwise, use decision scores.
    if hasattr(model, "predict_proba"):
        return model.predict_proba(X)[:, 1]
    return model.decision_function(X)


def evaluate(model, X, y, n_splits=5, gap=2000, name=None):
    """Evaluate a model using time-series cross-validation and return fold results."""

    # Preserve the chronological order of observations when creating folds.
    # The gap excludes observations between each training and validation set.
    cv = TimeSeriesSplit(n_splits=n_splits, gap=gap)
    rows = []

    for fold, (tr, va) in enumerate(cv.split(X), start=1):
        # Create a fresh copy of the model for each fold.
        m = clone(model)

        # Train the model and measure its training time.
        t0 = time.time()
        m.fit(X.iloc[tr], y.iloc[tr])
        fit_s = time.time() - t0

        # Score the validation orders and retrieve their actual outcomes.
        s = get_scores(m, X.iloc[va])
        yv = y.iloc[va]

        # Store performance metrics and training time for this fold.
        rows.append({
            "model": name,
            "fold": fold,
            "train_n": len(tr),
            "val_n": len(va),
            "val_late_rate": yv.mean(),
            "pr_auc": average_precision_score(yv, s),
            "roc_auc": roc_auc_score(yv, s),
            "recall_top5": recall_at_top(yv, s, 0.05),
            "recall_top10": recall_at_top(yv, s, 0.10),
            "recall_top20": recall_at_top(yv, s, 0.20),
            "fit_seconds": fit_s,
        })

    # Return one row per fold so models can be compared later.
    return pd.DataFrame(rows)


Below are our 2 baseline models as defined in ReadMe section 1.7:

**B0 Random ranking**: random scores, so PR-AUC is about the late rate of each fold.

**B1 Simple logistic regression**: promised days + customer state only (as defined in section 1.7).

In [14]:
class RandomRanker(BaseEstimator, ClassifierMixin):
    """No-skill baseline that assigns random scores to orders."""

    def __init__(self, random_state=RANDOM_STATE):
        # Set the random seed so scores are reproducible.
        self.random_state = random_state

    def fit(self, X, y):
        # Define the two possible classes: 0 = on time, 1 = late.
        # This baseline does not learn anything from the training data.
        self.classes_ = np.array([0, 1])
        return self

    def predict_proba(self, X):
        # Assign each order a random score between 0 and 1.
        # Higher scores mean higher predicted risk of late delivery.
        p = np.random.default_rng(self.random_state).random(len(X))

        # Return two columns: probability-like scores for on-time and late.
        return np.c_[1 - p, p]


# Preprocessing for the logistic regression baseline (B1).
b1_pre = ColumnTransformer([
    # Log-transform promised delivery days, then standardize the values.
    ("days", Pipeline([
        ("log", FunctionTransformer(np.log1p)),
        ("scale", StandardScaler())
    ]), ["promised_days_from_approval"]),

    # Convert customer states into numerical indicator columns.
    # Rare states are grouped when supported by the encoder.
    ("state", OneHotEncoder(
        handle_unknown="infrequent_if_exist",
        min_frequency=100
    ), ["customer_state"]),
], sparse_threshold=0)


# Store both baselines in a dictionary for convenient evaluation.
baselines = {
    # B0: Random scores; establishes a no-skill reference point.
    "B0 Random": RandomRanker(),

    # B1: Predict late-delivery risk using two simple features.
    "B1 LogReg (promised days + state)": Pipeline([
        ("pre", b1_pre),
        ("clf", LogisticRegression(
            max_iter=1000,
            random_state=RANDOM_STATE
        ))
    ]),
}

The code below does three main things:
- Reuses the same preprocessing: Each model gets the existing preprocess transformations through a pipeline.
- Defines seven candidate algorithms: Logistic regression, balanced logistic regression, Linear SVM, Gaussian Naive Bayes, Decision Tree, Random Forest, and Histogram-based Gradient Boosting.
- Sets model parameters: Controls how each algorithm learns, including model complexity and training behavior.

In [15]:
def with_pre(clf):
    # Apply the same preprocessing to every model for a fair comparison.
    return Pipeline([("pre", clone(preprocess)), ("clf", clf)])


candidates = {
    # Provides a simple, interpretable baseline for predicting late orders.
    "Logistic Regression": with_pre(LogisticRegression(
        C=1.0, max_iter=2000, random_state=RANDOM_STATE)),

    # Late orders are only ~7% of the data, so this model gives more weight
    # to late orders to test whether it improves detection of the minority class.
    "Logistic Regression (balanced)": with_pre(LogisticRegression(
        C=1.0, max_iter=2000, class_weight="balanced",
        random_state=RANDOM_STATE)),

    # Tests a different linear decision boundary to see whether it ranks
    # late orders better than logistic regression.
    "Linear SVM": with_pre(LinearSVC(
        C=0.1, max_iter=5000, random_state=RANDOM_STATE)),

    # Provides a fast, probability-based model with different assumptions,
    # allowing us to test whether a simple statistical approach works well.
    "Gaussian Naive Bayes": with_pre(GaussianNB()),

    # Captures nonlinear relationships and feature interactions using rules.
    # Depth and leaf-size limits help reduce overfitting.
    "Decision Tree": with_pre(DecisionTreeClassifier(
        max_depth=6, min_samples_leaf=100, random_state=RANDOM_STATE)),

    # Combines many decision trees to capture complex patterns while
    # reducing the instability and overfitting of a single tree.
    "Random Forest": with_pre(RandomForestClassifier(
        n_estimators=300, min_samples_leaf=20, max_features="sqrt",
        n_jobs=-1, random_state=RANDOM_STATE)),

    # Builds trees sequentially, with each new tree correcting earlier errors.
    # Tests whether boosting can identify complex patterns missed by other models.
    "Gradient Boosting (HistGB)": with_pre(HistGradientBoostingClassifier(
        learning_rate=0.05, max_iter=300, max_leaf_nodes=31, min_samples_leaf=50,
        early_stopping=False, random_state=RANDOM_STATE)),
}

Next, we run all baseline and candidate models through the same cross-validation evaluation process and combine their results into one table.

In [17]:
# Combine the baseline models and candidate models into one dictionary.
all_models = {**baselines, **candidates}

# Store the evaluation results from each model.
fold_results = []

# Evaluate each model using the same training data and cross-validation setup.
for name, model in all_models.items():
    print(f"Running {name} ...")

    # Run time-series cross-validation and save the results for every fold.
    fold_results.append(
        evaluate(model, X_train_sel, y_train, name=name)
    )

# Combine all model results into one DataFrame for comparison.
# Each row represents one model evaluated on one fold.
fold_df = pd.concat(fold_results, ignore_index=True)

Running B0 Random ...
Running B1 LogReg (promised days + state) ...
Running Logistic Regression ...
Running Logistic Regression (balanced) ...
Running Linear SVM ...
Running Gaussian Naive Bayes ...
Running Decision Tree ...
Running Random Forest ...
Running Gradient Boosting (HistGB) ...


The following code summarizes the cross-validation results for all nine models into one benchmark table. It:

Contains the mean and standard deviation for PR-AUC, ROC-AUC, and recall at the top 5%, 10%, and 20%, formatted as mean ± standard deviation.

Sorts the models by mean PR-AUC, from highest to lowest.

In [18]:
# Metrics used to compare the models across validation folds.
metrics = ["pr_auc", "recall_top5", "recall_top10", "recall_top20", "roc_auc"]

# Group results by model so we can summarize performance across its folds.
g = fold_df.groupby("model", sort=False)

# Calculate the average and standard deviation for each metric.
mean, std = g[metrics].mean(), g[metrics].std()

# Create a benchmark table with one row per model.
bench = pd.DataFrame(index=mean.index)

# Display each metric as mean ± standard deviation across folds.
for m in metrics:
    bench[m] = [
        f"{a:.3f} ± {b:.3f}"
        for a, b in zip(mean[m], std[m])
    ]

# Add the average model training time in seconds per fold.
bench["fit_s (mean)"] = g["fit_seconds"].mean().round(1)

# Temporarily use mean PR-AUC to rank models from best to worst.
bench["_sort"] = mean["pr_auc"]
bench = bench.sort_values("_sort", ascending=False).drop(columns="_sort")

# Rename columns to make the final table easier to read.
bench.columns = [
    "PR-AUC", "Recall top 5%", "Recall top 10%",
    "Recall top 20%", "ROC-AUC", "Fit s/fold"
]

# Display the final benchmark table.
bench

,PR-AUC,Recall top 5%,Recall top 10%,Recall top 20%,ROC-AUC,Fit s/fold
model,,,,,,
Logistic Regression,0.158 ± 0.122,0.115 ± 0.017,0.215 ± 0.027,0.385 ± 0.032,0.667 ± 0.032,0.8
Logistic Regression (balanced),0.157 ± 0.118,0.113 ± 0.018,0.215 ± 0.024,0.380 ± 0.030,0.664 ± 0.031,1.9
Linear SVM,0.155 ± 0.121,0.104 ± 0.019,0.208 ± 0.031,0.377 ± 0.039,0.663 ± 0.033,0.7
B1 LogReg (promised days + state),0.150 ± 0.107,0.099 ± 0.033,0.201 ± 0.051,0.364 ± 0.049,0.646 ± 0.035,0.2
Random Forest,0.141 ± 0.107,0.104 ± 0.011,0.186 ± 0.022,0.348 ± 0.042,0.638 ± 0.029,18.1
Gradient Boosting (HistGB),0.139 ± 0.109,0.098 ± 0.017,0.171 ± 0.036,0.318 ± 0.058,0.620 ± 0.049,7.1
Gaussian Naive Bayes,0.119 ± 0.089,0.075 ± 0.012,0.148 ± 0.010,0.304 ± 0.025,0.603 ± 0.025,0.3
Decision Tree,0.110 ± 0.084,0.086 ± 0.015,0.154 ± 0.021,0.290 ± 0.042,0.569 ± 0.034,0.5
B0 Random,0.088 ± 0.067,0.051 ± 0.008,0.099 ± 0.009,0.193 ± 0.007,0.501 ± 0.007,0.0


Shortlisting:

Rule: Rank models by mean PR-AUC lift across validation folds, using each fold’s late-order rate as the baseline.

Select 3–5 models that perform well and represent different approaches (linear, tree-based ensemble, probabilistic) for further comparison.:

In [21]:
# Calculate PR-AUC lift relative to each fold's late-order rate.
# A lift above 1 means performance is better than random ranking.
fold_df["pr_auc_lift"] = (
    fold_df["pr_auc"] / fold_df["val_late_rate"]
)

# Summarize normalized performance across the five validation folds.
lift_summary = (
    fold_df.groupby("model")
    .agg(
        mean_pr_auc_lift=("pr_auc_lift", "mean"),
        std_pr_auc_lift=("pr_auc_lift", "std"),
        mean_pr_auc=("pr_auc", "mean"),
        mean_recall_top10=("recall_top10", "mean")
    )
    .sort_values("mean_pr_auc_lift", ascending=False)
)

print("Model ranking by mean PR-AUC lift:")
display(lift_summary.round(3))

# Shortlist three models with different learning approaches.
shortlist_models = [
    "Logistic Regression",
    "Random Forest",
    "Gradient Boosting (HistGB)"
]

shortlist = lift_summary.loc[shortlist_models].sort_values(
    "mean_pr_auc_lift", ascending=False
)

print("\nFinal shortlist:")
display(shortlist.round(3))


Model ranking by mean PR-AUC lift:


,mean_pr_auc_lift,std_pr_auc_lift,mean_pr_auc,mean_recall_top10
model,,,,
Logistic Regression,1.808,0.214,0.158,0.215
Logistic Regression (balanced),1.805,0.190,0.157,0.215
B1 LogReg (promised days + state),1.762,0.348,0.150,0.201
Linear SVM,1.756,0.262,0.155,0.208
Random Forest,1.614,0.155,0.141,0.186
Gradient Boosting (HistGB),1.538,0.243,0.139,0.171
Gaussian Naive Bayes,1.374,0.112,0.119,0.148
Decision Tree,1.253,0.158,0.110,0.154
B0 Random,0.997,0.030,0.088,0.099



Final shortlist:


,mean_pr_auc_lift,std_pr_auc_lift,mean_pr_auc,mean_recall_top10
model,,,,
Logistic Regression,1.808,0.214,0.158,0.215
Random Forest,1.614,0.155,0.141,0.186
Gradient Boosting (HistGB),1.538,0.243,0.139,0.171
